## ADC Linker LSTM Model

In [102]:
# following this tutorial on LSTMs in pytorch: https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/

In [103]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as TF
import numpy as np
import pandas as pd
import random
import re
import math
from sklearn.model_selection import KFold

In [104]:
#add seed for reproducible results
seed = 42

#python
random.seed(seed)

#numpy
np.random.seed(seed)

#pytorch
torch.manual_seed(seed)

#for GPU
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

#make PyTorch behavior deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [105]:
df = pd.read_pickle("data/adc_data_complete_v2.pkl")

In [106]:
df['smiles'].isnull().any()

np.False_

In [107]:
df.head()

,index_x,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Product name,smiles,calc_SA_score,TPSA,QED,LogP,...,amino,phosphate,total_rings,aromatic_rings,aliphatic_rings,num_h_donors,num_h_acceptors,FractionCSP3,num_rot_bonds,Surface Area
0,0,0,0,0,(Ac)Phe-Lys(Alloc)-PABC-PNP,C=CCOC(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC(...,3.508295,204.30,0.036588,4.5636,...,0,0,3,3,0,4,10,0.285714,18,288.060675
1,2,2,2,2,Fmoc-Val-Cit-PAB,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.338596,171.88,0.163403,3.6140,...,0,0,4,3,1,6,6,0.333333,13,256.107099
2,3,3,3,3,Fmoc-Val-Cit-PAB-PNP,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.753739,230.32,0.030495,5.7455,...,0,0,5,4,1,5,10,0.275000,16,321.776490
3,4,4,4,4,Mc-Val-Cit-PABC-PNP,CC(C)[C@H](NC(=O)CCCCCN1C(=O)C=CC1=O)C(=O)N[C@...,3.796268,258.47,0.032958,2.8084,...,0,0,3,2,1,5,11,0.400000,20,304.496626
4,5,5,5,5,Val-cit-PAB-OH,CC(C)[C@H](N)C(=O)N[C@@H](CCCNC(N)=O)C(=O)Nc1c...,2.890504,159.57,0.315935,0.0340,...,1,0,1,1,0,6,5,0.500000,10,158.418876


In [108]:
# SMILES tokenizer reference: https://deepchem.readthedocs.io/en/2.4.0/api_reference/tokenizers.html

SMI_REGEX = re.compile(r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|B|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])")
smiles_tokens = df['smiles'].str.findall(SMI_REGEX)

# smiles tokens is a series of lists for all patterns found per smiles string
# explode converts into a single series with all pattern matches
unique_tokens = smiles_tokens.explode().unique()
smiles_map = {"PAD": 0}
smiles_map_rest = {char: i+1 for i,char in enumerate(unique_tokens)}
smiles_map.update(smiles_map_rest)
#smiles_map

In [109]:
# convert smiles to their integer representation example

smiles_ex = 'CC(=O)N[C@@H](CC1=CC=CC=C1)C(=O)N[C@@H](CCCCNC(=O)OCC=C)C(=O)NC2=CC=C(C=C2)COC(=O)OC3=CC=C(C=C3)[N+](=O)[O-]'
smiles_char_tokens = SMI_REGEX.findall(smiles_ex)
smiles_token_seq = [smiles_map[char] for char in smiles_char_tokens]
#smiles_char_seq

In [110]:
# test set up for embedding layer for pytorch to handles these tokens

embedding_dim = 16 # this is num dimensions that LSTM will use to describe each smiles character

embedding_layer = nn.Embedding(num_embeddings=len(unique_tokens), embedding_dim=embedding_dim, padding_idx=smiles_map["PAD"])
embedding_layer.weight.shape #30 unique smiles patterns by 16 dimensions per pattern

torch.Size([30, 16])

In [111]:
# test embedding on our smiles example string

input_tensor = torch.LongTensor(smiles_token_seq) # this is just our token representation converted to tensor
embedded_seq = embedding_layer(input_tensor) # now each token is represented by three dimension vector of random initialized weights
#embedded_seq

In [112]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.embedding = nn.Embedding(
            num_embeddings=len(smiles_map),
            embedding_dim=input_dim,
            padding_idx=smiles_map["PAD"],
        )
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True) 
        self.fc = nn.Linear(hidden_dim, output_dim)

        #added dropout to reduce overfitting
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        embedded_seq = self.embedding(x)
        out, (hn, cn) = self.lstm(embedded_seq)
        out = out[:, -1, :] # Take last time step
        out = self.dropout(out) #apply dropout even when layer_dim = 1
        out = self.fc(out)  #final layer
        return out

In [113]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [114]:
def smiles_to_tokens(smile_pattern_list):
    return [smiles_map[char] for char in smile_pattern_list]


def collate_smiles(batch):
    # this adds the padding, short smiles will get padded to length of longest smile in batch
    smiles = [item[0] for item in batch]
    scores = [item[1] for item in batch]

    smiles_padded = pad_sequence(
        smiles, batch_first=True, padding_value=smiles_map["PAD"]
    )

    scores_stack = torch.stack(
        scores
    )  # scores is a list from batch, stack reshapes to a size of (batch,1)

    return smiles_padded, scores_stack


class adcDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.file = pd.read_pickle(directory)
        self.smiles = df["smiles"].str.findall(SMI_REGEX).apply(smiles_to_tokens)
        self.SA_score = torch.tensor(np.array((self.file["calc_SA_score"])))

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = torch.tensor(self.smiles[idx])
        score = torch.tensor([self.SA_score[idx]])

        return smile, score

In [115]:
dataset = adcDataset("data/adc_data_complete_v2.pkl")

In [116]:
#k-fold cross validation
#5 fold split: 80/20
kfolds = 5
kf = KFold(n_splits = kfolds, shuffle = True, random_state = 42)

#store validation results
fold_losses = []

#for avg_percent_error
fold_models = []       
fold_val_indices = []



for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"\n -- Fold {fold+1} / {kfolds} -- ")

    #make two subsets
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    #dataloader
    train_loader = DataLoader(train_subset, batch_size = 64, shuffle = True, collate_fn = collate_smiles)
    val_loader = DataLoader(val_subset, batch_size = 64, shuffle = False, collate_fn = collate_smiles)

    #reset model for each fold
    model = LSTMModel(input_dim = 128, hidden_dim = 256, layer_dim = 1, output_dim = 1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
    criterion = nn.MSELoss()

    #training loop
    num_epochs = 10
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0

        for i, (smile, score) in enumerate(train_loader):
            smile, score = smile.to(device), score.to(device)

            optimizer.zero_grad()
            output = model(smile)
            loss = criterion(output.float(), score.float())
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} / {num_epochs}, Train Loss: {avg_train_loss:.4f}")

    #validation loop
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for smile, score in val_loader:
            smile, score = smile.to(device), score.to(device)
            output = model(smile)
            loss = criterion(output.float(), score.float())
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    fold_losses.append(avg_val_loss)
    print(f"Fold {fold+1} Validation Loss: {avg_val_loss:.4f}")

    fold_models.append(model)
    fold_val_indices.append(val_idx)


print("\n -- K-Fold Results --")
print("Fold Losses:", fold_losses)
mse = np.mean(fold_losses)
print(f"MSE Validation Loss: {mse:.4f}")
rmse = np.sqrt(mse)
print(f"RMSE Validation Loss: {rmse:.4f}")


 -- Fold 1 / 5 -- 
Epoch 1 / 10, Train Loss: 9.8625
Epoch 2 / 10, Train Loss: 7.6576
Epoch 3 / 10, Train Loss: 0.7256
Epoch 4 / 10, Train Loss: 0.5866
Epoch 5 / 10, Train Loss: 0.5808
Epoch 6 / 10, Train Loss: 0.5649
Epoch 7 / 10, Train Loss: 0.5489
Epoch 8 / 10, Train Loss: 0.5271
Epoch 9 / 10, Train Loss: 0.4692
Epoch 10 / 10, Train Loss: 0.3954
Fold 1 Validation Loss: 0.3290

 -- Fold 2 / 5 -- 
Epoch 1 / 10, Train Loss: 9.7763
Epoch 2 / 10, Train Loss: 7.6422
Epoch 3 / 10, Train Loss: 0.7506
Epoch 4 / 10, Train Loss: 0.5921
Epoch 5 / 10, Train Loss: 0.5779
Epoch 6 / 10, Train Loss: 0.5553
Epoch 7 / 10, Train Loss: 0.5414
Epoch 8 / 10, Train Loss: 0.5144
Epoch 9 / 10, Train Loss: 0.4076
Epoch 10 / 10, Train Loss: 0.3622
Fold 2 Validation Loss: 0.3416

 -- Fold 3 / 5 -- 
Epoch 1 / 10, Train Loss: 9.6102
Epoch 2 / 10, Train Loss: 7.8204
Epoch 3 / 10, Train Loss: 0.7683
Epoch 4 / 10, Train Loss: 0.6237
Epoch 5 / 10, Train Loss: 0.6000
Epoch 6 / 10, Train Loss: 0.6107
Epoch 7 / 10, Trai

In [117]:
SA_predictions = []
SA_true = []

for fold, val_idx in enumerate(fold_val_indices):
    val_subset = Subset(dataset, val_idx)
    val_loader = DataLoader(val_subset, batch_size = 64, shuffle = False, collate_fn = collate_smiles)

    model = fold_models[fold]
    model.eval()
    
    with torch.no_grad():
        for smile, score in val_loader:
            smile, score = smile.to(device), score.to(device)
            preds = model(smile)

            SA_predictions.extend(preds.cpu().numpy())
            SA_true.extend(score.cpu().numpy())

# convert from tensor to array for predicted and true scores
SA_predictions = np.array(SA_predictions).reshape(-1)
SA_true = np.array(SA_true).reshape(-1)

avg_percent_error = np.mean(np.abs((SA_true - SA_predictions) / (SA_true + 1e-8)) * 100)
print(f"Mean Absolute Percent Error: {avg_percent_error:.2f}%")

Mean Absolute Percent Error: 14.13%
